In [4]:
import sys

sys.path.append('../../')

In [14]:
import pandas as pd

df = pd.read_parquet(DATA_RAW_DIR / "final_data_cleaner.parquet")

In [15]:
df.isnull().sum()

text     0
label    0
dtype: int64

In [ ]:
df_new = df.dropna()

In [16]:
from ml_factory.datasets.precompute_tokens import precompute_tokens
from ml_factory.utils import merge_parts_to_dir
from ml_factory import DATA_RAW_DIR, DATA_PROCESSED_DIR

In [17]:
from transformers import AutoTokenizer

In [18]:
data_path = DATA_RAW_DIR / 'final_data_cleaner.parquet'
output_path = DATA_PROCESSED_DIR / '32_128_processed'

paths = precompute_tokens(dataset_path=data_path, 
                          out_path=output_path, 
                          tokenizer=AutoTokenizer,
                          pretrained_tokenizer="google/bert_uncased_L-2_H-128_A-2",
                          save_bytes=100_000_000,
                          token_chunk=True, 
                          stride=32, 
                          max_length=128)

Wrote 97731 items to c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed.part0.npz
Wrote 97890 items to c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed.part1.npz
Wrote 97836 items to c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed.part2.npz
Wrote 97693 items to c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed.part3.npz
Wrote 27184 items to c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed.part4.npz
Wrote 5 part files. Concatenate with merge_parts() before use.
Saved tokenizer config path, in path c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_facto

In [19]:
merge_parts_to_dir(part_paths=paths,
                   out_dir=output_path)

Merged 5 parts into directory c:\Users\Jivesh Gawde\Documents\NMIMS\sem3\FortexAI\Backend\ml_factory\notebooks\..\..\ml_factory\data\processed\32_128_processed


WindowsPath('c:/Users/Jivesh Gawde/Documents/NMIMS/sem3/FortexAI/Backend/ml_factory/notebooks/../../ml_factory/data/processed/32_128_processed')

In [20]:
from ml_factory.datasets import PromptBERTDataset

In [21]:
dataset = PromptBERTDataset(output_path)

In [25]:
dataset[4]

PromptBERTResult(tokenized=tensor([  101,  2009,  1005,  1055,  1037,  5658,  4651,  1010,  1999,  1037,
         3168,  1010,  2000,  1996,  2142,  2163,  1997,  3097,  4681,  1010,
         1000,  2002,  5275,  1010, 13384,  2008,  2043,  2060,  3032,  5309,
         2149,  6363,  1999,  2344,  2000,  2224,  2068,  2006,  2248,  6089,
         1006,  2107,  2004,  2005,  1996,  9343,  1998,  4855,  1997, 11540,
         1007,  1010,  2027,  6464,  2507,  1996,  2149,  1037,  5717,  1011,
         3037,  5414,  1517,  2823,  2012,  2335,  2043,  2027,  2064,  2560,
         8984,  2009,  1012,  2358,  8004, 24725,  2081,  2010,  7928,  2004,
         2132,  1997,  1037,  2142,  3741,  5997,  1997, 22171,  3228, 11433,
         2000,  4769,  1996,  3795,  3361,  5325,  1012,  1005,  1010,  6251,
         2475,  1024,  1005,  3312,  2358,  8004, 24725,  2003,  1996,  3453,
         1997,  1037, 10501,  3396,  1999,  5543,  1012,  1005,   102,     0,
            0,     0,     0,     0,  

In [30]:
import numpy as np
a = np.load(DATA_PROCESSED_DIR / '32_128_processed/num_chunks.npy')
b = np.load(DATA_PROCESSED_DIR / '32_128_processed/ids.npy')

In [63]:
count = {}
seen_text = set()
for i,j in zip(a,b):
    if j not in seen_text:
        count[i] = count.get(i, 0) + 1
    seen_text.add(j)

print(count)

{np.int64(1): 314398, np.int64(2): 45114, np.int64(7): 72, np.int64(4): 437, np.int64(3): 2051, np.int64(5): 235, np.int64(20): 4, np.int64(6): 109, np.int64(10): 46, np.int64(9): 55, np.int64(12): 25, np.int64(11): 38, np.int64(24): 1, np.int64(8): 66, np.int64(15): 9, np.int64(14): 7, np.int64(21): 4, np.int64(19): 4, np.int64(18): 7, np.int64(25): 4, np.int64(13): 11, np.int64(16): 8, np.int64(29): 4, np.int64(26): 1, np.int64(31): 1, np.int64(51): 1, np.int64(27): 1, np.int64(28): 1}


In [68]:
for i, j in zip(a,b):
    if i == 51:
        print(j)
        break

7c452d5977c9ea8ec9e29151f15c1a30064f3f9d1a3a812f4a9d084d51ec4dad


In [38]:
import polars as pl
from ml_factory.utils import give_id_to_data
data = pl.read_parquet(DATA_RAW_DIR / 'final_data_cleaner.parquet')

data = give_id_to_data(data)

In [69]:
s = data.filter(pl.col('id') == '7c452d5977c9ea8ec9e29151f15c1a30064f3f9d1a3a812f4a9d084d51ec4dad')['text'][0]

In [70]:
s

'.. --. -. --- .-. .  .- .-.. .-..  - .... .  .. -. ... - .-. ..- -.-. - .. --- -. ...  -.-- --- ..-  --. --- -  -... . ..-. --- .-. . .-.-.-  ..-. .-. --- --  -. --- .--  --- -. --..--  -.-- --- ..-  .- .-. .  --. --- .. -. --.  - ---  .- -.-. -  .- ...  -.-. .... .- - --. .--. -  .-- .. - ....  -.. . ...- . .-.. --- .--. . .-.  -- --- -.. .  . -. .- -... .-.. . -.. .-.-.-  .- ...  -.-- --- ..- .-.  -.- -. --- .-- .-.. . -.. --. .  .. ...  -.-. ..- -  --- ..-. ..-.  .. -.  ..--- ----- ..--- .---- --..--  -.-- --- ..-  .--. .-. --- -... .- -... .-.. -.--  -.. --- -. .----. -  -.- -. --- .--  .-- .... .- -  - .... .- -  .. ... .-.-.-  ..  .-- .. .-.. .-..  --. .. ...- .  -.-- --- ..-  .-  -... .-. .. . ..-.  ... ..- -- -- .- .-. -.--  .- -... --- ..- -  .. - .-.-.-  .. -  .-- .- ...  .. -. - .-. --- -.. ..- -.-. . -..  .. -.  -- .. -..  ..--- ----- ..--- ..---  .- -. -..  .. -  .-- .- ...  -.-. .-. . .- - . -..  .- ...  .-  -- . .- -. ...  - ---  - . ... -  .. -. - . .-. -. .- .-..  -..

In [44]:
tokenizer = AutoTokenizer.from_pretrained("google/bert_uncased_L-2_H-128_A-2")

In [59]:
re = tokenizer(s, max_length=128, stride=32, padding="max_length", truncation=True, return_overflowing_tokens=True, save_bytes=2000)

In [60]:
re['overflow_to_sample_mapping']

[0]

In [61]:
re

{'input_ids': [[101, 6251, 2487, 1024, 1005, 2522, 25855, 24619, 14543, 20704, 3207, 17240, 1010, 2747, 2006, 2604, 1996, 14719, 2686, 2276, 1010, 2038, 2468, 1996, 2711, 2007, 1996, 6493, 2561, 2994, 1999, 2686, 1012, 2002, 2038, 5119, 2098, 2039, 6273, 2487, 2420, 1012, 1005, 1010, 6251, 2475, 1024, 1005, 12513, 2907, 2501, 2005, 6493, 2994, 1999, 2686, 1012, 1005, 4339, 1996, 15792, 21527, 2011, 1996, 2206, 3793, 1012, 3437, 2007, 3893, 2030, 4997, 1024, 4629, 4038, 1010, 2214, 1011, 13405, 6071, 3185, 12483, 2015, 1010, 1998, 10218, 2540, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [71]:
from ml_factory.datasets.sampler import SplitSampler

In [72]:
sampler = SplitSampler(train_split=0.7, test_split=0.15, validation_split=0.15, seed=444)
sampler.build_split(dataset.item_ids, dataset.labels)

In [73]:
sampler.save_split(DATA_PROCESSED_DIR / 'splitsampler_train_7_test_15.pt')

In [74]:
from torch.utils.data import Subset, DataLoader

In [75]:
train_dataset = Subset(dataset, dataset.ids_to_position(sampler.get_split('train')))
val_dataset = Subset(dataset, dataset.ids_to_position(sampler.get_split('validation')))
test_dataset = Subset(dataset, dataset.ids_to_position(sampler.get_split('test')))


In [76]:
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=100)
val_loader = DataLoader(val_dataset, shuffle=False, batch_size=100)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=100)



In [77]:
for data in train_loader:
    print(data)
    break

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'ml_factory.datasets.PromptBERTResult'>